# 3  Predicting rejected tax registrations

**Goal:** predict whether a tax registration will be rejected, so staff can check it early, and read the reason for one decision.

**Data:** 4,000 synthetic tax registrations; about one in five was rejected.

**The data used here:**
- `tax_registrations.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/tax_registrations.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/tax_registrations.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

## Step 1  The registrations

In [ ]:
#@title Step 1  Load the registrations
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
df = pd.read_csv(DATA + "tax_registrations.csv")
INPUTS = ["business_activity", "legal_form", "years_in_operation", "declared_turnover_aed", "documents_missing", "trade_licence_valid", "submitted_via_online_portal", "name_mismatch"]
X = pd.get_dummies(df[INPUTS], dtype=int); y = df["rejected"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
test_rows = df.loc[X_test.index]
big((f"{len(df):,}", "registrations"), (f"{y.mean():.0%}", "rejected", RED))
table(df[INPUTS + ["rejected"]], {(i, "rejected"): "#fbd9d9" for i in range(8) if df.loc[i, "rejected"] == 1}, n=8)
box("What you see", "The target is a <b>category</b>: rejected (1) or accepted (0). The model learns from past registrations whose outcome is known.", "info")

## Step 2  Logistic regression: a probability

In [ ]:
#@title Step 2  Train it and look at eight registrations
logit = LogisticRegression(max_iter=3000).fit(X_train, y_train)
proba = logit.predict_proba(X_test)[:, 1]
view = test_rows[["registration_id", "documents_missing", "trade_licence_valid", "name_mismatch"]].head(8).copy()
view["probability of rejection"] = proba[:8].round(2); view["actually rejected"] = y_test.values[:8]
display(view)
box("What you see", "For each registration the model gives a <b>probability</b> between 0 and 1. A threshold turns it into a decision: above it, staff check the registration early.", "info")

## Step 3  The threshold is a choice

In [ ]:
#@title Step 3  Move the threshold  (then move the slider)
def show(threshold):
    flag = proba >= threshold
    fig, ax = plt.subplots(figsize=(10, 2.2))
    ax.scatter(proba, np.random.default_rng(0).random(len(proba)), c=np.where(flag, RED, "#bbbbbb"), s=10)
    ax.axvline(threshold, color=INK, lw=3); ax.set_yticks([]); ax.set_xlabel("probability of rejection, one dot per test registration"); plt.show()
    big((int(flag.sum()), "flagged for an early check"), (f"{accuracy_score(y_test, flag):.1%}", "decisions right"), (int((flag & (y_test.values == 1)).sum()), "of them really rejected", GREEN))
    box("Do you see it?", "Same model, different decisions. A low threshold means more early checks and more work; a high one means fewer checks and more surprises. "
        "<b>The service chooses the threshold</b>, based on what each kind of mistake costs.", "see")
_ = live(show, threshold=fslider("Threshold", 0.05, 0.95, 0.05, 0.5))

## Step 4  A decision tree: rules you can read

In [ ]:
#@title Step 4  Draw the tree
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
plt.figure(figsize=(18, 7)); plot_tree(tree, feature_names=X.columns, class_names=["accepted", "rejected"], filled=True, fontsize=10, impurity=False); plt.show()
box("How to read it", "Each box asks one question. Left means <i>yes</i> (the condition is true), right means <i>no</i>. The colour shows which outcome is more common in that group.", "info")

In [ ]:
#@title Step 4b  Follow one registration through the tree  (pick a registration on the right, then click &#9654;)
registration = 1 #@param {type:"slider", min:1, max:20, step:1}
one = X_test.iloc[[registration - 1]]; row = test_rows.iloc[registration - 1]
steps = []
for node in tree.decision_path(one).indices:
    f = tree.tree_.feature[node]
    if f >= 0:
        name, val, thr = X.columns[f], one.iloc[0, f], tree.tree_.threshold[node]
        base = next((c for c in INPUTS if name.startswith(c + "_")), None)
        if base is not None:
            cat = name[len(base) + 1:]; others = [str(c) for c in df[base].dropna().unique() if str(c) != cat]
            shown = cat if val == 1 else (others[0] if len(others) == 1 else "not " + cat)
            steps.append(f"<b>{base.replace('_', ' ')}</b> is {shown}")
        else: steps.append(f"<b>{name.replace('_', ' ')}</b> is {val:g} ({'at most' if val <= thr else 'above'} {thr:g})")
pred = ["accepted", "rejected"][tree.predict(one)[0]]
box(f"Registration {row['registration_id']}: likely {pred}", "<br>".join(f"{i + 1}. {s}" for i, s in enumerate(steps)) +
    f"<br><br>Actually: <b>{'rejected' if row['rejected'] else 'accepted'}</b>", "see" if pred == "rejected" else "ok")

## Step 5  Depth again

In [ ]:
#@title Step 5  Move the slider  (then move the slider)
def show(depth):
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    a, b = accuracy_score(y_train, t.predict(X_train)), accuracy_score(y_test, t.predict(X_test))
    big((depth, "depth"), (f"{a:.1%}", "right on training data"), (f"{b:.1%}", "right on test data", RED))
    if a - b > 0.05: box("Do you see it?", "A growing gap: the tree fits the training data better and better, but gets worse on new registrations. Limiting the depth keeps it useful.", "see")
_ = live(show, depth=islider("Depth of the tree", 1, 25, 3))

In [ ]:
#@title Step 6  Which model would you show a business owner?  (choose, then click &#9654;)
choice = "Choose..." #@param ["Choose...", "Logistic regression", "Decision tree, depth 3", "Decision tree, depth 25"]
acc_lr, acc_tree = accuracy_score(y_test, proba >= 0.5), accuracy_score(y_test, tree.predict(X_test))
big((f"{acc_lr:.1%}", "logistic regression"), (f"{acc_tree:.1%}", "tree, depth 3"))
_ = answer(choice, {
    "Logistic regression": (False, f"It is the more accurate model here ({acc_lr:.1%} against {acc_tree:.1%}), and the better choice when accuracy matters most. But its reasons are a list of weights that are hard to explain to a business owner."),
    "Decision tree, depth 3": (True, f"A little less accurate ({acc_tree:.1%} against {acc_lr:.1%}), but every decision is a short rule you can say out loud, such as 'the trade licence was not valid'. For a conversation with a business owner, that matters more."),
    "Decision tree, depth 25": (False, "Hundreds of rules, and worse on new registrations. Neither explainable nor better."),
})

## Result

In [ ]:
#@title Result
box("Do you see it now?", "The model now flags registrations that are likely to be rejected, <b>so staff can contact the business before it waits weeks for a refusal</b>. "
    "You set the threshold yourself and saw the rule behind one decision. Both are choices the service owns, not the model.", "see")

---
## Stretch (optional, for those who code)

In [ ]:
# Stretch: change the threshold of the logistic regression in code and describe the effect on flagged registrations and accuracy
for t in [0.3, 0.5, 0.7]:
    print(t, "flagged:", int((proba >= t).sum()), "accuracy:", round(accuracy_score(y_test, proba >= t), 3))